In [3]:
!pip install pandas nltk
import pandas as pd
import nltk
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\shahi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\shahi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [6]:
df = pd.read_csv("../data/email_evaluation_dataset_mohammadshahidulla.csv")
df.head()

,id,email_text,expected_action,expected_tone
0,1,This is a weekly newsletter with general updat...,notify,neutral
1,2,"Congratulations, you have been selected for th...",notify,urgent
2,3,Your order has been shipped and will be delive...,notify,polite
3,4,Reminder: The project review meeting is schedu...,notify,urgent
4,5,Your order has been shipped and will be delive...,ignore,urgent


In [7]:
import pandas as pd
import numpy as np
import re

In [9]:
df['clean_text'] = (
    df['email_text']
    .str.lower()
    .str.replace('[^a-zA-Z ]', '', regex=True)
)

df[['email_text', 'clean_text']].head()


,email_text,clean_text
0,This is a weekly newsletter with general updat...,this is a weekly newsletter with general updat...
1,"Congratulations, you have been selected for th...",congratulations you have been selected for the...
2,Your order has been shipped and will be delive...,your order has been shipped and will be delive...
3,Reminder: The project review meeting is schedu...,reminder the project review meeting is schedul...
4,Your order has been shipped and will be delive...,your order has been shipped and will be delive...


In [ ]:
def email_assistant(clean_text):
    text = clean_text.lower()
    if "urgent" in text or "submit" in text or "deadline" in text:
        return "urgent"
    elif "thank you" in text:
        return "polite"
    else:
        return "neutral"

df['predicted_tone'] = df['clean_text'].apply(email_assistant)
df[['email_text', 'predicted_tone']].head()


,email_text,expectedtone
0,This is a weekly newsletter with general updat...,neutral
1,"Congratulations, you have been selected for th...",neutral
2,Your order has been shipped and will be delive...,neutral
3,Reminder: The project review meeting is schedu...,neutral
4,Your order has been shipped and will be delive...,neutral


In [ ]:
def email_assistant(clean_text):
    text = clean_text.lower()
    # action rules
    if "invoice" in text or "payment" in text or "submit" in text:
        action = "respond"
    elif "security" in text or "login" in text or "account" in text:
        action = "notify"
    else:
        action = "ignore"
    # tone rules
    if "urgent" in text or "deadline" in text or "immediately" in text:
        tone = "urgent"
    elif "thank you" in text or "please" in text:
        tone = "polite"
    else:
        tone = "neutral"
    return action, tone


results = df['clean_text'].apply(email_assistant)
df['predicted_action'] = results.apply(lambda x: x[0])
df['predicted_tone']   = results.apply(lambda x: x[1])
df[['email_text', 'predicted_action', 'predicted_tone']].head()

,email_text,predicted_action,predicted_tone
0,This is a weekly newsletter with general updat...,ignore,neutral
1,"Congratulations, you have been selected for th...",ignore,neutral
2,Your order has been shipped and will be delive...,ignore,neutral
3,Reminder: The project review meeting is schedu...,ignore,neutral
4,Your order has been shipped and will be delive...,ignore,neutral


In [ ]:
df['action_correct'] = df['predicted_action'] == df['expected_action']
df['tone_correct'] = df['predicted_tone'] == df['expected_tone']

In [23]:
action_accuracy = df['action_correct'].mean() * 100
tone_accuracy = df['tone_correct'].mean() * 100

print(f"Action Accuracy: {action_accuracy:.2f}%")
print(f"Tone Accuracy: {tone_accuracy:.2f}%")

Action Accuracy: 37.00%
Tone Accuracy: 30.00%


In [24]:
df_errors = df[df['action_correct'] == False][
    ['email_text', 'expected_action', 'predicted_action']
]
df_errors.head()


,email_text,expected_action,predicted_action
0,This is a weekly newsletter with general updat...,notify,ignore
1,"Congratulations, you have been selected for th...",notify,ignore
2,Your order has been shipped and will be delive...,notify,ignore
3,Reminder: The project review meeting is schedu...,notify,ignore
5,Security alert: A login attempt was detected f...,respond,notify


In [39]:
df.head()

,id,email_text,expected_action,expected_tone,clean_text,predicted_action,predicted_tone,action_correct,tone_correct
0,1,This is a weekly newsletter with general updat...,notify,neutral,this is a weekly newsletter with general updat...,ignore,neutral,False,True
1,2,"Congratulations, you have been selected for th...",notify,urgent,congratulations you have been selected for the...,ignore,neutral,False,False
2,3,Your order has been shipped and will be delive...,notify,polite,your order has been shipped and will be delive...,ignore,neutral,False,False
3,4,Reminder: The project review meeting is schedu...,notify,urgent,reminder the project review meeting is schedul...,ignore,neutral,False,False
4,5,Your order has been shipped and will be delive...,ignore,urgent,your order has been shipped and will be delive...,ignore,neutral,True,False


In [ ]:
#Which type of emails were hardest to classify?
#Informational and newsletter-style emails.

#Why did your rules fail in some cases?
#The rule-based system relies on keyword matching. 

#How could an LLM improve this process? 
#An LLM can understand context.

In [40]:
# 6. Save results for submission
OUT = "../data/milestone2_evaluation_output.csv"
df.to_csv(OUT, index=False)
print("Saved:", OUT)

Saved: ../data/milestone2_evaluation_output.csv
